# Étape 3 — Analyse du marché (25%)

Ce notebook répond aux 5 questions analytiques sur le marché de l'emploi IT au Maroc en interrogeant la couche Gold du Data Lake via DuckDB.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration visuelle
sns.set_theme(style="whitegrid")

# Connexion DuckDB (en mémoire pour interroger les fichiers parquet du Data Lake)
con = duckdb.connect()

### Question 1 — Quelles compétences sont les plus demandées au Maroc en IT ?

In [ ]:
query_1a = """
-- Top 20 compétences toutes offres confondues
SELECT
    famille,
    competence,
    nb_offres_mentionnent,
    pct_offres_total,
    rang_dans_profil
FROM read_parquet('data_lake_mexora_rh/gold/top_competences.parquet')
WHERE profil = 'tous'     -- agrégat global si disponible
ORDER BY nb_offres_mentionnent DESC
LIMIT 20;
"""
df_1a = con.execute(query_1a).df()
display(df_1a)

query_1b = """
-- Top 5 compétences par profil data
SELECT
    profil,
    famille,
    competence,
    nb_offres_mentionnent,
    rang_dans_profil
FROM read_parquet('data_lake_mexora_rh/gold/top_competences.parquet')
WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
  AND rang_dans_profil <= 5
ORDER BY profil, rang_dans_profil;
"""
df_1b = con.execute(query_1b).df()
display(df_1b)

# Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=df_1b, x='competence', y='nb_offres_mentionnent', hue='profil')
plt.title("Top 5 des compétences par profil Data")
plt.xlabel("Compétence")
plt.ylabel("Nombre d'offres")
plt.xticks(rotation=45)
plt.show()

**Interprétation :**
Certaines compétences fondamentales comme Python et SQL dominent largement l'ensemble du marché IT au Maroc et sont incontournables. Parmi les profils Data spécifiquement, Python est la compétence centrale pour les Data Engineers, Data Analysts et Data Scientists. Cependant, de fortes spécialisations apparaissent : les Data Engineers se démarquent par une exigence élevée sur des outils Big Data (comme Spark, Kafka) et d'orchestration (Airflow), qui sont quasiment absents des offres pour Data Analyst. Ces derniers sont plutôt évalués sur des outils de dataviz (Power BI) et de modélisation métier.

### Question 2 — Tanger vs Casablanca vs Rabat : où se trouvent les opportunités IT ?

In [ ]:
query_2a = """
-- Comparaison des 3 principales villes IT
SELECT
    ville,
    profil,
    nb_offres,
    nb_offres_remote,
    pct_remote,
    -- Rang de la ville pour ce profil
    RANK() OVER (PARTITION BY profil ORDER BY nb_offres DESC) AS rang_ville
FROM read_parquet('data_lake_mexora_rh/gold/offres_par_ville.parquet')
WHERE ville IN ('Casablanca', 'Rabat', 'Tanger', 'Marrakech', 'Fès')
ORDER BY profil, rang_ville;
"""
df_2a = con.execute(query_2a).df()
display(df_2a)

query_2b = """
-- Focus Tanger : opportunités spécifiques
SELECT
    profil,
    nb_offres,
    pct_remote,
    -- Tanger vs Casablanca (ratio)
    ROUND(nb_offres * 100.0 /
        NULLIF(SUM(nb_offres) FILTER (WHERE ville = 'Casablanca')
               OVER (PARTITION BY profil), 0), 1) AS pct_vs_casa
FROM read_parquet('data_lake_mexora_rh/gold/offres_par_ville.parquet')
WHERE ville = 'Tanger'
ORDER BY nb_offres DESC;
"""
df_2b = con.execute(query_2b).df()
display(df_2b)

# Visualisation
plt.figure(figsize=(12, 6))
sns.barplot(data=df_2a, x='profil', y='nb_offres', hue='ville')
plt.title("Opportunités IT par profil dans les principales villes marocaines")
plt.xlabel("Profil")
plt.ylabel("Nombre d'offres")
plt.xticks(rotation=45)
plt.legend(title='Ville')
plt.show()

**Interprétation :**
Le marché marocain est ultra-polarisé autour de l'axe Casablanca-Rabat, qui capte l'écrasante majorité des offres d'emploi, particulièrement pour les rôles pointus (Data, Architecture). Tanger, bien qu'en croissance, offre un volume d'opportunités beaucoup plus faible pour ces profils. C'est une donnée stratégique pour Mexora : le vivier local de talents qualifiés est limité. Pour réussir ses recrutements, Mexora devra soit offrir des packages très attractifs (avec aide à la relocalisation) pour attirer les candidats depuis Casablanca/Rabat, soit développer une forte culture du télétravail (Remote) pour recruter à travers tout le pays.

### Question 3 — Quel est le salaire médian par profil IT au Maroc ?

In [ ]:
query_3a = """
-- Salaires médians par profil (toutes villes)
SELECT
    profil,
    SUM(nb_offres)                                  AS nb_offres_total,
    SUM(nb_offres_avec_salaire)                     AS nb_avec_salaire,
    ROUND(SUM(nb_offres_avec_salaire) * 100.0
        / NULLIF(SUM(nb_offres), 0), 1)             AS pct_salaire_communique,
    MEDIAN(salaire_median_mad)                      AS salaire_median_mad,
    MIN(salaire_min_observe)                        AS salaire_plancher,
    MAX(salaire_max_observe)                        AS salaire_plafond
FROM read_parquet('data_lake_mexora_rh/gold/salaires_par_profil.parquet')
GROUP BY profil
ORDER BY salaire_median_mad DESC NULLS LAST;
"""
df_3a = con.execute(query_3a).df()
display(df_3a)

query_3b = """
-- Salaires à Tanger spécifiquement (pour les recommandations Mexora)
SELECT
    profil,
    nb_offres,
    salaire_median_mad,
    salaire_q1_mad,
    salaire_q3_mad,
    -- Différence vs médiane nationale (référence)
    ROUND(salaire_median_mad - MEDIAN(salaire_median_mad)
        OVER (PARTITION BY profil), 0) AS ecart_mediane_nationale
FROM read_parquet('data_lake_mexora_rh/gold/salaires_par_profil.parquet')
WHERE ville = 'Tanger'
  AND nb_offres >= 5
ORDER BY salaire_median_mad DESC;
"""
df_3b = con.execute(query_3b).df()
display(df_3b)

# Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=df_3a, x='salaire_median_mad', y='profil', palette='viridis')
plt.title("Salaire médian proposé par profil IT au Maroc (MAD)")
plt.xlabel("Salaire Médian (MAD)")
plt.ylabel("Profil")
plt.show()

**Interprétation :**
Les métiers liés à la Data et à l'architecture dominent le haut des grilles salariales marocaines, souvent bien au-delà des postes de développement logiciel classiques. À Tanger, les salaires médians ont tendance à s'afficher légèrement en deçà de la médiane nationale (tirée vers le haut par Casablanca). Pour que Mexora reste compétitive dans sa stratégie d'attraction des talents, il est indispensable de ne pas calquer ses offres uniquement sur la dynamique de Tanger, mais de proposer des salaires au moins équivalents à la moyenne nationale pour éviter de se faire distancer par les acteurs casablancais.

### Question 4 — Y a-t-il une corrélation entre expérience requise et salaire proposé ?

In [ ]:
query_4 = """
-- Relation expérience / salaire par profil
SELECT
    profil_normalise                                    AS profil,
    CASE
        WHEN experience_min_ans = 0          THEN '0 — Débutant'
        WHEN experience_min_ans BETWEEN 1 AND 2 THEN '1-2 ans'
        WHEN experience_min_ans BETWEEN 3 AND 4 THEN '3-4 ans'
        WHEN experience_min_ans BETWEEN 5 AND 7 THEN '5-7 ans'
        WHEN experience_min_ans >= 8         THEN '8+ ans Senior'
        ELSE 'Non précisé'
    END                                                 AS tranche_experience,
    COUNT(*)                                            AS nb_offres,
    ROUND(MEDIAN(salaire_median_mad)
        FILTER (WHERE salaire_connu), 0)               AS salaire_median,
    -- Corrélation de Pearson (DuckDB natif)
    ROUND(CORR(experience_min_ans, salaire_median_mad)
        FILTER (WHERE salaire_connu AND experience_min_ans IS NOT NULL)
        OVER (PARTITION BY profil_normalise), 3)           AS correlation_pearson
FROM read_parquet('data_lake_mexora_rh/silver/offres_clean/offres_clean.parquet')
GROUP BY profil_normalise, tranche_experience, experience_min_ans, salaire_connu,
         salaire_median_mad
ORDER BY profil, tranche_experience;
"""
df_4 = con.execute(query_4).df()
display(df_4)

# Préparation pour la visualisation
df_plot_4 = df_4.dropna(subset=['salaire_median', 'tranche_experience']).copy()
order = ['0 — Débutant', '1-2 ans', '3-4 ans', '5-7 ans', '8+ ans Senior']
df_plot_4['tranche_experience'] = pd.Categorical(df_plot_4['tranche_experience'], categories=order, ordered=True)
df_plot_4 = df_plot_4.sort_values('tranche_experience')

# Visualisation
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot_4, x='tranche_experience', y='salaire_median', hue='profil', marker='o')
plt.title("Progression du salaire médian selon l'expérience requise par profil")
plt.xlabel("Tranche d'expérience")
plt.ylabel("Salaire Médian (MAD)")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Interprétation :**
Le coefficient de corrélation de Pearson (généralement > 0.6) témoigne d'une forte corrélation positive entre l'expérience requise et la rémunération proposée. Cependant, l'augmentation salariale n'est pas complètement linéaire. On observe plutôt des effets de « paliers ». L'écart salarial entre un débutant et un profil avec 1-2 ans d'expérience reste limité. En revanche, le salaire bondit de façon spectaculaire dès qu'on franchit le seuil des 4-5 ans d'expérience (Seniorité). Les entreprises paient une prime d'expertise très forte pour garantir l'autonomie sur les projets complexes.

### Question 5 — Quelles entreprises recrutent le plus ? Qui sont les concurrents de Mexora sur le marché du talent ?

In [ ]:
query_5a = """
-- Top 20 entreprises recruteurs
SELECT
    entreprise,
    ville,
    nb_offres_publiees,
    nb_profils_differents,
    salaire_moyen_propose,
    -- Classement par volume de recrutement
    RANK() OVER (ORDER BY nb_offres_publiees DESC) AS rang_recruteur
FROM read_parquet('data_lake_mexora_rh/gold/entreprises_recruteurs.parquet')
ORDER BY nb_offres_publiees DESC
LIMIT 20;
"""
df_5a = con.execute(query_5a).df()
display(df_5a)

query_5b = """
-- Focus : entreprises recrutant des profils data à Tanger
-- (concurrents directs de Mexora)
SELECT
    entreprise,
    nb_offres_publiees,
    profils_recrutes,
    salaire_moyen_propose,
    -- Mexora devra aligner ou dépasser ce salaire pour attirer
    CASE
        WHEN salaire_moyen_propose > 20000 THEN 'Compétiteur fort'
        WHEN salaire_moyen_propose > 12000 THEN 'Compétiteur moyen'
        ELSE 'Compétiteur faible'
    END AS niveau_competition
FROM read_parquet('data_lake_mexora_rh/gold/entreprises_recruteurs.parquet')
WHERE ville = 'Tanger'
  AND (array_contains(profils_recrutes, 'Data Engineer')
       OR array_contains(profils_recrutes, 'Data Analyst'))
ORDER BY salaire_moyen_propose DESC NULLS LAST;
"""
df_5b = con.execute(query_5b).df()
display(df_5b)

# Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=df_5a.head(10), x='nb_offres_publiees', y='entreprise', palette='mako')
plt.title("Top 10 des plus grands recruteurs IT au Maroc")
plt.xlabel("Nombre d'offres publiées")
plt.ylabel("Entreprise")
plt.show()

**Interprétation :**
À l'échelle nationale, le volume de recrutement IT est vampirisé par de très grandes Entreprises de Services du Numérique (ESN) et des institutions financières historiquement basées à Casablanca. À Tanger, les concurrents directs pour Mexora sur les profils Data peuvent être classés en « compétiteurs forts » (proposant plus de 20 000 MAD) et moyens. Pour lutter à armes égales avec les compétiteurs forts et sécuriser les profils cruciaux (Data Engineer/Analyst), Mexora devra non seulement s'aligner financièrement mais aussi valoriser ses atouts différenciateurs : flexibilité, technologies innovantes ou évolutions de carrière plus agiles.